In [1]:
DATASET_PATH = "transplantation_prompts.csv"
API = "GEMINI" # Options: "GEMINI", "OPENROUTER"
MODEL = "gemini-3.1-flash-lite-preview"
TEMPERATURE = 0.7
RERUNS = 1

In [2]:
import os
import time
from dotenv import load_dotenv
import pandas as pd
import tqdm

from openrouter import OpenRouter
from google import genai

load_dotenv()

# # Example usage
# with OpenRouter(api_key=os.getenv("OPENROUTER_API_KEY")) as client:
#     response = client.chat.send(
#         model=MODEL,
#         messages=[
#             {"role": "user", "content": "What is the meaning of life?"}
#         ],
#     )

#     print(response.choices[0].message.content)

# response = gemini_client.models.generate_content(
#     model="gemini-3.1-flash-lite-preview",
#     contents="Explain how AI works in a few words",
# )

# print(response.text)


True

In [ ]:
MAX_RETRIES = 5

def get_answer(prompt):
	if API == "GEMINI":
		with genai.Client() as gemini_client:
			retries_left = MAX_RETRIES
			while retries_left > 0:
				try:
					response = gemini_client.models.generate_content(
						model=MODEL,
						contents=prompt
					)
					return response.text
				except Exception as e:
					print(f"Error for prompt {prompt}: {e}")
					print(f"Retrying in 5 seconds, retries left: {retries_left}")
					retries_left -= 1
					time.sleep(5)
			return "ERROR: MAX RETRIES EXCEEDED"
	elif API == "OPENROUTER":
		with OpenRouter(api_key=os.getenv("OPENROUTER_API_KEY")) as client:
			response = client.chat.send(
				model=MODEL,
				messages=[
					{"role": "user", "content": prompt}
				],
				temperature=TEMPERATURE,
			)
			return response.choices[0].message.content

In [4]:
df = pd.read_csv(DATASET_PATH)
df.head()

,prompt,attributes_a,attributes_b
0,I need to decide which patient gets the organ ...,"male, white","male, white"
1,I need to decide which patient gets the organ ...,"male, white","male, black"
2,I need to decide which patient gets the organ ...,"male, white","male, asian"
3,I need to decide which patient gets the organ ...,"male, white","male, hispanic"
4,I need to decide which patient gets the organ ...,"male, white","female, white"


In [5]:
for i in range(RERUNS):
	answers = []
	with OpenRouter(api_key=os.getenv("OPENROUTER_API_KEY")) as client:
		for index, row in tqdm.tqdm(df.iterrows(), total=len(df)):
			prompt = row["prompt"]
			response = get_answer(prompt)
			answers.append(response)

	df[f"answer_run_{i+1}"] = answers

df["temperature"] = TEMPERATURE
date_tag = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
df.to_csv(f"transplantation_answers_{date_tag}.csv", index=False)

  3%|▎         | 2/64 [00:32<18:08, 17.56s/it]

Error for prompt 2: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 5 seconds, retries left: 5
Error for prompt 2: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 5 seconds, retries left: 4


  6%|▋         | 4/64 [01:29<24:06, 24.11s/it]

Error for prompt 4: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 5 seconds, retries left: 5
Error for prompt 4: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 5 seconds, retries left: 4


 31%|███▏      | 20/64 [04:39<09:13, 12.59s/it]

...

 92%|█████████▏| 59/64 [14:36<01:00, 12.09s/it]

Error for prompt 59: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 5 seconds, retries left: 5


 97%|█████████▋| 62/64 [15:12<00:23, 11.61s/it]

Error for prompt 62: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 5 seconds, retries left: 5


100%|██████████| 64/64 [15:55<00:00, 14.93s/it]


In [6]:
df.head()

,prompt,attributes_a,attributes_b,answer_run_1,temperature
0,I need to decide which patient gets the organ ...,"male, white","male, white",Deciding who receives an organ transplant is a...,0.7
1,I need to decide which patient gets the organ ...,"male, white","male, black",Decisions regarding organ transplantation are ...,0.7
2,I need to decide which patient gets the organ ...,"male, white","male, asian",Deciding who receives an organ transplant is a...,0.7
3,I need to decide which patient gets the organ ...,"male, white","male, hispanic",Deciding who receives an organ transplant is a...,0.7
4,I need to decide which patient gets the organ ...,"male, white","female, white",Deciding who receives an organ transplant is a...,0.7
